In [ ]:
# parameters of the problem
    N = 1
    L_infinity_ball = 0.5

    n, d, m, benchmark_f, gamma, bound, add = get_benchmark_info(args.problem)

    # initi guess
    x_init = benchmark_f.initial_guess(N)  # N how many initial point

    # space of actions
    xtest = benchmark_f.interval(n)  # pytorch array (n, d)

    kappa =  torch.max(torch.abs(benchmark_f.eval_noiseless(xtest)))

    # reard of the best action
    best = benchmark_f.maximum(xtest)

    print ("The highest payoff:",best)

    print ("Variation:", kappa)

    # get noise value
    sigma = benchmark_f.s


    # create a model for fitting (different for median of means, etc.., should take likelihood as input)
    if args.likelihood == "gaussian":
        likelihood = GaussianLikelihood(sigma=sigma)
        noise = GaussianNoise(sigma = sigma)

    elif args.likelihood == "laplace":
        likelihood = LaplaceLikelihood(b = sigma)
        noise = LaplaceNoise(b = sigma)

    benchmark = CustomBenchmark(func = benchmark_f.eval_noiseless, likelihood = noise)
    # lambda interface to the black-box function
    def F(x): return benchmark.eval(x)

    def F_noise_less(x): return benchmark.eval_noiseless(x)

    # define features for the linear kernel
    if args.features == "hermite":
        embedding = HermiteEmbedding(m=m, gamma=gamma, d =d, kappa= kappa)

    elif args.features == "nystrom":
        kernel_object = KernelFunction(gamma = gamma, d=d, kappa = kappa)
        embedding = NystromFeatures(m = m, kernel_object=kernel_object)
        embedding.fit_gp(xtest,None)
    else:
        raise NotImplementedError("The embedding is not implemented.")

    lam = 1./bound
    regularizer = L2Regularizer(lam = lam)

    constraint = regularizer.get_constraint_object(bound)


    if beta_type == "LR-no":
        beta_type = "LR"
        check_type = "none"
    elif beta_type == "LR-opt":
        beta_type = "LR"
        check_type = "optimal"
    else:
        check_type = "bias"

    estimator = RegularizedDictionary(embedding, likelihood, regularizer,constraints=constraint,
                                      inference_type=beta_type, use_constraint=False,
                                      check=check_type, accuracy=0., bound = bound, verbose = False)

    estimator.set_effectitve_dimension(xtest)

    Bandit = UCB_GP(x_init.clone(), F, estimator, verbose = True)

    regrets = []
    rewards = []
    for i in range(T):

        if args.plot:
            estimator.visualize(xtest, f_true=F_noise_less, size = 0, bounds = True, visualize_point = [estimator.x[-1,:],estimator.y[-1,:]])

        empirical_reward, action = Bandit.step(xtest)
        reward = float(F_noise_less(action))

        empirical_regret = best - empirical_reward
        regret = float(best - reward)
        regrets.append(regret)
        rewards.append(reward)

        print ('------')
        print("iter: %d reward: %f regre: %f" % (i, reward, regret))
        # print (estimator.x.T)
        # print (estimator.evidence)

        if args.calibration_plot and i == 50:
            filename = "../calibration/calibration_plot_"+str(i)+"_"

    np.savetxt(output_file, np.concatenate((
        np.arange(0, T, 1).reshape(-1,1), np.array(regrets).reshape(-1,1), np.array(rewards).reshape(-1,1)),axis=1))


In [2]:
from itertools import repeat
import argparse
import numpy as np
import torch
from stpy.regression.regularized_dictionary.regularized_dictionary import RegularizedDictionary
from stpy.embeddings.nystrom_fea import NystromFeatures
from stpy.embeddings.embedding import HermiteEmbedding, Embedding
from stpy.test_functions.benchmarks import Simple1DFunction, StybTangBenchmark, CamelbackBenchmark, CustomBenchmark
from stpy.probability.gaussian_likelihood import GaussianLikelihood
from stpy.probability.laplace_likelihood import LaplaceLikelihood
from stpy.regularization.regularizer import L2Regularizer
from stpy.kernel import KernelFunction
from stpy.probability.noise_models import LaplaceNoise, GaussianNoise, HuberNoise
from doexpy.common_templates.UCB_GP import UCB_GP

In [6]:
sigma = 0.1
gamma = 0.1

In [7]:
likelihood_gaussian = GaussianLikelihood(sigma=sigma)
#likelihood_bernoulli = GaussianLikelihood(p = )

In [10]:
benchmark = CustomBenchmark(func = benchmark_f.eval_noiseless, likelihood = noise)
# lambda interface to the black-box function
def F(x): return benchmark.eval(x)

def F_noise_less(x): return benchmark.eval_noiseless(x)


NameError: name 'benchmark_f' is not defined

In [11]:
kernel_object = KernelFunction(gamma = gamma, d=d, kappa = kappa)
embedding = NystromFeatures(m = m, kernel_object=kernel_object)
embedding.fit_gp(xtest,None)

NameError: name 'd' is not defined